# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 3 – Data Extraction** (simplified, beginner version)

Pipeline position: **3 of 8**

## Objective
The product **name** holds useful facts as text, e.g. *"LG 1.5 Ton 5 Star Inverter Split AC"*. We pull them out into new columns: `asin`, `tonnage`, `energy_star`, `brand`, `ac_type`, `is_inverter`, `has_wifi`.

## Step 1 – Setup

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


In [6]:
df = pd.read_csv("/content/Air_Conditioners_Filtered.csv")
print("Loaded:", df.shape)
df[["name", "ratings", "discount_price"]].head()

Loaded: (32, 9)


,name,ratings,discount_price
0,Helium Smart Choice Air Smart Inverter Split A...,3.3,22990.0
1,IFB 0.95 Ton 3 Star 2026 AI Powered Inverter S...,4.4,30990.0
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",4.9,34990.0
3,"Acerpure Chill Neo 1.5 Ton 3 star, 5280W Max C...",4.4,27490.0
4,"Hitachi Smart Choice 1.5 Ton 3 Star, New Star ...",4.1,36500.0


## Step 2 – Product id (ASIN)
The id is the 10 characters after `/dp/` in the link.

In [7]:
df["asin"] = df["link"].str.split("/dp/").str[1].str[:10]
df[["asin"]].head()

,asin
0,B0GZVYF7N5
1,B0GSJV6XTF
2,B0H87QTXQC
3,B0GL2LWCVL
4,B0GP71KXP5


## Step 3 – Tonnage
`str.extract()` finds a pattern in text. The pattern `(\d+\.?\d*)\s*ton` means:
- `\d+` one or more digits, `\.?` an optional dot, `\d*` more digits  (e.g. `1.5`)
- `\s*` optional spaces
- `ton` the word ton

The part inside `( )` is what we keep. We use the lower-case name so *Ton*, *TON* and *ton* all match.

In [8]:
name_lower = df["name"].str.lower()

df["tonnage"] = name_lower.str.extract(r"(\d+\.?\d*)\s*ton")[0].astype(float)
df["tonnage"].value_counts(dropna=False).sort_index()

,count
tonnage,
0.50,1
0.80,3
0.95,2
1.00,6
1.42,2
1.50,17
2.00,1


## Step 4 – Energy star rating
Same idea: one digit followed by the word *star* (e.g. `5 star` or `5-star`).

In [9]:
df["energy_star"] = name_lower.str.extract(r"(\d)\s*-?\s*star")[0].astype(float)
df["energy_star"].value_counts(dropna=False).sort_index()

,count
energy_star,
2.0,2
3.0,23
4.0,2
5.0,5


## Step 5 – Brand
1. First guess: the first word of the name.
2. Some brands need special spelling (LG, IFB, MarQ) or have two words (Blue Star, O General). We fix them using a dictionary: `"text in name": "correct brand name"`.

In [10]:
# step 1: first word of the name
df["brand"] = df["name"].str.split().str[0].str.strip(",").str.title()

# step 2: special brands
special_brands = {"blue star": "Blue Star", "bluestar": "Blue Star", "o general": "O General", "ogeneral": "O General",
                  "lg": "LG", "ifb": "IFB", "tcl": "TCL", "marq": "MarQ"}

for key, correct_name in special_brands.items():
    starts_with_brand = name_lower.str.startswith(key + " ") | name_lower.str.startswith(key + ",")
    df.loc[starts_with_brand, "brand"] = correct_name

df["brand"].value_counts()

,count
brand,
Blue Star,4
Hitachi,4
LG,4
Samsung,3
Daikin,3
Voltas,2
Cruise,2
IFB,2
Lloyd,1


## Step 6 – AC type, inverter and Wi-Fi
Simple word checks with `str.contains()`.

In [11]:
df["ac_type"] = "Other"
df.loc[name_lower.str.contains("split"), "ac_type"] = "Split"
df.loc[name_lower.str.contains("window"), "ac_type"] = "Window"

df["is_inverter"] = name_lower.str.contains("inverter") & ~name_lower.str.contains("fixed speed")
df["has_wifi"] = name_lower.str.contains("wifi|wi-fi")

print(df["ac_type"].value_counts())
print()
print("Inverter models:", df["is_inverter"].sum(), "of", len(df))
print("Wi-Fi models   :", df["has_wifi"].sum(), "of", len(df))

ac_type
Split    31
Other     1
Name: count, dtype: int64

Inverter models: 31 of 32
Wi-Fi models   : 7 of 32


## Step 7 – Check the result

In [12]:
df[["name", "brand", "tonnage", "energy_star", "ac_type", "is_inverter"]].head(10)

,name,brand,tonnage,energy_star,ac_type,is_inverter
0,Helium Smart Choice Air Smart Inverter Split A...,Helium,0.80,3.0,Split,True
1,IFB 0.95 Ton 3 Star 2026 AI Powered Inverter S...,IFB,0.95,3.0,Split,True
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",Blue Star,1.50,3.0,Split,True
3,"Acerpure Chill Neo 1.5 Ton 3 star, 5280W Max C...",Acerpure,1.50,3.0,Split,True
4,"Hitachi Smart Choice 1.5 Ton 3 Star, New Star ...",Hitachi,1.50,3.0,Split,True
5,"Sharp 0.95 Ton 3 star, New Star Rated,Split AC...",Sharp,0.95,3.0,Split,True
6,"LG Essential Series, 0.8 Ton 2 Star, Smart Inv...",LG,0.80,2.0,Split,True
7,"Lloyd 1.5 Ton 3 Star Inverter Split AC, 2026 N...",Lloyd,1.50,3.0,Split,True
8,"Samsung 1 Ton 3 Star,New Star rated Bespoke AI...",Samsung,1.00,3.0,Split,True
9,"LG 1.5 Ton 3 Star, Smart Inverter Split AC (10...",LG,1.50,3.0,Split,True


In [13]:
checked = ["asin", "brand", "tonnage", "energy_star"]
found = df[checked].notna().sum()
print("Values found out of", len(df), "listings:")
print(found)
print()
print("Coverage %:")
print((found / len(df) * 100).round(1))

Values found out of 32 listings:
asin           32
brand          32
tonnage        32
energy_star    32
dtype: int64

Coverage %:
asin           100.0
brand          100.0
tonnage        100.0
energy_star    100.0
dtype: float64


## Step 8 – Numeric and text columns, and views

In [14]:
print("Numeric columns    :", df.select_dtypes(include="number").columns.tolist())
print("Text columns       :", df.select_dtypes(include="object").columns.tolist())
df.describe().round(2)

Numeric columns    : ['ratings', 'no_of_ratings', 'discount_price', 'actual_price', 'tonnage', 'energy_star']
Text columns       : ['name', 'main_category', 'sub_category', 'image', 'link', 'asin', 'brand', 'ac_type']


,ratings,no_of_ratings,discount_price,actual_price,tonnage,energy_star
count,32.00,32.00,31.00,31.00,32.00,32.00
mean,4.02,313.34,34483.13,50970.68,1.29,3.31
std,0.41,321.97,7635.09,22270.28,0.33,0.82
min,3.10,12.00,16999.00,12470.00,0.50,2.00
25%,3.78,160.00,30140.00,35870.00,1.00,3.00
50%,4.05,214.00,34990.00,50990.00,1.50,3.00
75%,4.22,300.50,37490.00,68095.00,1.50,3.00
max,4.90,1400.00,49804.00,90790.00,2.00,5.00


In [15]:
price_view = df[["name", "brand", "tonnage", "energy_star", "discount_price", "actual_price"]]
rating_view = df[["name", "brand", "ratings", "no_of_ratings", "discount_price"]]
print("price_view  :", price_view.shape)
print("rating_view :", rating_view.shape)
price_view.head()

price_view  : (32, 6)
rating_view : (32, 5)


,name,brand,tonnage,energy_star,discount_price,actual_price
0,Helium Smart Choice Air Smart Inverter Split A...,Helium,0.80,3.0,22990.0,40100.0
1,IFB 0.95 Ton 3 Star 2026 AI Powered Inverter S...,IFB,0.95,3.0,30990.0,50990.0
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",Blue Star,1.50,3.0,34990.0,17495.0
3,"Acerpure Chill Neo 1.5 Ton 3 star, 5280W Max C...",Acerpure,1.50,3.0,27490.0,13745.0
4,"Hitachi Smart Choice 1.5 Ton 3 Star, New Star ...",Hitachi,1.50,3.0,36500.0,63990.0


## Step 9 – Save

In [17]:
df.to_csv("Air_Conditioners_Extracted.csv", index=False)
print("Saved:", df.shape)

Saved: (32, 16)


**Output of Stage 3:** `Air_Conditioners_Extracted.csv`.  **Next → Stage 4: Data Validation & Cleaning.**